# 02 — DAPT Training
Domain-Adaptive Continued Pre-Training on the chunked legal corpus. Run `01` first.

**Runtime:** T4 GPU, required. **This is the long-running step** — set `save_steps` low in `configs/dapt_config.yaml` (already set to 200) so a session disconnect doesn't cost you the whole run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = "/content/drive/MyDrive/legal-compliance-slm"

In [ ]:
!git clone https://github.com/Shankar-behera/legal-compliance-slm.git
%cd legal-compliance-slm
!pip install -r requirements.txt -q

In [ ]:
import os, shutil
os.makedirs("data/processed", exist_ok=True)
# pull processed data back from Drive if this is a fresh runtime
for fname in ["dapt_chunks.jsonl"]:
    src = f"{DRIVE_ROOT}/data/processed/{fname}"
    if os.path.exists(src):
        shutil.copy(src, f"data/processed/{fname}")
        print("restored", fname)
    else:
        print("MISSING:", src, "— run notebook 01 first")

## Secrets

Use Colab's Secrets manager (key icon, left sidebar) — never hardcode tokens.

In [ ]:
from google.colab import userdata
import os, wandb

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
wandb.login(key=userdata.get("WANDB_API_KEY"))

## Confirm GPU

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

## Run DAPT

Reads `configs/dapt_config.yaml` — batch=2, grad_accum=8 (effective batch 16), lr=2e-4 cosine, 1 epoch. Checkpoints every 200 steps to the Drive-mounted `output_dir` set in the config.

In [ ]:
from src.training.train_dapt import run_dapt

final_checkpoint_dir = run_dapt("configs/dapt_config.yaml")
print("DAPT checkpoint:", final_checkpoint_dir)

## Plot the loss curve

In [ ]:
from src.training.plot_curves import plot_curve
import glob

# trainer_state.json lives in the most recent checkpoint dir
state_path = sorted(glob.glob(f"{final_checkpoint_dir}/../checkpoint-*/trainer_state.json"))[-1]
plot_curve(state_path, output_path="docs/training_curve_dapt.png", title="DAPT Loss (Qwen2.5-1.5B)")

## Resuming after a disconnect

If the session dropped mid-run, point `train_dapt.py` at the last checkpoint instead of restarting from the base model:
```python
from transformers import TrainingArguments
# re-run run_dapt(...) — HF Trainer auto-resumes from output_dir's latest
# checkpoint-N/ folder as long as output_dir is unchanged (it's on Drive,
# so it survives the disconnect).
```
